In [1]:
import os
import torch
import torch.nn as nn
from torchvision.transforms import v2 as transforms
import librosa
import numpy as np
from sklearn.metrics import roc_auc_score
import wandb

In [2]:
generator = torch.Generator().manual_seed(42)
np.random.seed(42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [4]:
transform = transforms.RandomCrop(size=(128, 256))


class AudioDataset(torch.utils.data.Dataset):
    def __init__(self, audio_dir, train):
        self.audio_dir = audio_dir
        file_list = os.listdir(audio_dir)

        labels = np.zeros(len(file_list), dtype=int) if train else [1 if el[0] == 'a' else 0 for el in file_list]
        self.labels = torch.tensor(labels, dtype=torch.int8).to(device)

        loads = [librosa.load(os.path.join(audio_dir, el), sr=None) for el in file_list]
        spectrograms = [librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=128) for audio, sr in loads]
        spect_dbs = [
            torch.tensor(
                librosa.power_to_db(spec, ref=np.max),
                dtype=torch.float32
            )
            for spec in spectrograms
        ]

        spect_dbs = torch.stack(spect_dbs)

        mean = spect_dbs.mean(dim=0)
        std = spect_dbs.std(dim=0)

        self.spect_dbs = (spect_dbs - mean) / std

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return transform(self.spect_dbs[idx]), self.labels[idx]

In [5]:
train_dataset = AudioDataset('/kaggle/input/dcase-aml/dev_data/dev_data/slider/train', train=True)
test_dataset = AudioDataset('/kaggle/input/dcase-aml/dev_data/dev_data/slider/test', train=False)

In [6]:
batch_size = 16

train_set, validation_set = torch.utils.data.random_split(train_dataset, [0.8, 0.2], generator=generator)

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True
)
validation_loader = torch.utils.data.DataLoader(
    validation_set,
    batch_size=batch_size,
    shuffle=False
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [7]:
class CNNAE(nn.Module):
    def __init__(self):
        super(CNNAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, stride=1, padding=2),   # (32, 128, 256)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                                     # (32, 64, 128)
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # (64, 64, 128)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                                     # (64, 32, 64)
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),  # (128, 32, 64)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                                     # (128, 16, 32)
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2),   # (64, 32, 64)
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),    # (32, 64, 128)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=2, stride=2),     # (1, 128, 256)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.encoder(x)
        x = self.decoder(x)
        return x.squeeze(1)


class C1DNNAE(nn.Module):
    def __init__(self):
        super(C1DNNAE, self).__init__()
        # Encoder: input shape (batch, 128, 256)
        self.encoder = nn.Sequential(
            nn.Conv1d(128, 64, kernel_size=5, stride=2, padding=2),   # (batch, 64, 128)
            nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=5, stride=2, padding=2),    # (batch, 32, 64)
            nn.ReLU(),
            nn.Conv1d(32, 16, kernel_size=5, stride=2, padding=2),    # (batch, 16, 32)
            nn.ReLU(),
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(16, 32, kernel_size=4, stride=2, padding=1),  # (batch, 32, 64)
            nn.ReLU(),
            nn.ConvTranspose1d(32, 64, kernel_size=4, stride=2, padding=1),  # (batch, 64, 128)
            nn.ReLU(),
            nn.ConvTranspose1d(64, 128, kernel_size=4, stride=2, padding=1),  # (batch, 128, 256)
        )

    def forward(self, x):
        # x: (batch, 128, 256) expected
        x = self.encoder(x)
        x = self.decoder(x)
        return x


class C1DNNAE_INV(nn.Module):
    def __init__(self):
        super(C1DNNAE_INV, self).__init__()
        # Encoder: input shape (batch, 256, 128)
        self.encoder = nn.Sequential(
            nn.Conv1d(256, 128, kernel_size=5, stride=2, padding=2),   # (batch, 128, 64)
            nn.ReLU(),
            nn.Conv1d(128, 64, kernel_size=5, stride=2, padding=2),    # (batch, 64, 32)
            nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=5, stride=2, padding=2),     # (batch, 32, 16)
            nn.ReLU(),
        )
        # Decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(32, 64, kernel_size=4, stride=2, padding=1),   # (batch, 64, 32)
            nn.ReLU(),
            nn.ConvTranspose1d(64, 128, kernel_size=4, stride=2, padding=1),  # (batch, 128, 64)
            nn.ReLU(),
            nn.ConvTranspose1d(128, 256, kernel_size=4, stride=2, padding=1),  # (batch, 256, 128)
        )

    def forward(self, x):
        # x: (batch, 128, 256) expected
        x = x.permute(0, 2, 1)
        x = self.encoder(x)
        x = self.decoder(x)
        return x.permute(0, 2, 1)


class LAE(nn.Module):
    def __init__(self):
        super(LAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(128 * 256, 2048),
            nn.ReLU(),
            nn.Linear(2048, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

        self.decoder = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, 2048),
            nn.ReLU(),
            nn.Linear(2048, 128 * 256),
            nn.Tanh()
        )

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.encoder(x)
        x = self.decoder(x)
        return x.view(x.size(0), 128, 256)

In [8]:
def train_one_epoch(model, data_loader, optimizer, criterion, scheduler):
    model.train()

    total_loss = 0

    for inputs, _ in data_loader:
        inputs = inputs.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        # print(inputs.shape)
        # print(outputs.shape)

        loss = criterion(outputs, inputs)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()

    return total_loss / len(data_loader)


def validate(model, data_loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for inputs, _ in data_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, inputs)
            total_loss += loss.item()

    return total_loss / len(data_loader)


def compute_reconstruction_errors(model, data_loader):
    model.eval()
    errors = []
    with torch.no_grad():
        for batch, _ in data_loader:
            batch = batch.to(device)
            reconstructed = model(batch)
            # print(reconstructed.shape, batch.shape)
            error = torch.mean(((reconstructed - batch) ** 2).reshape(batch.size(0), -1), dim=1)
            # print(error.shape)
            errors.extend(error.cpu().numpy())
    return errors


def test(model, data_loader):
    model.eval()

    errors = compute_reconstruction_errors(model, data_loader)

    roc_auc_scores = roc_auc_score(test_dataset.labels.cpu(), errors)
    print(f"ROC AUC Score test: {roc_auc_scores}")

    return roc_auc_scores

In [9]:
def train(lr, step_size, gamma, epochs, use_wandb=False):
    model = C1DNNAE().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma)

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scheduler)
        val_loss = validate(model, validation_loader, criterion)
        roc_auc_score = test(model, test_loader)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

        if use_wandb:
            wandb.log({
                "train_loss": train_loss,
                "val_loss": val_loss,
                "roc_auc_score": roc_auc_score,
                "epoch": epoch + 1,
            })

    return model

In [10]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
key = user_secrets.get_secret('wandb-api-key')

wandb.login(key=key)

sweep_configuration = {
    "method": "grid",
    "metric": {"goal": "maximize", "name": "roc_auc_score"},
    'name': "convolutional_1d_autoencoder",
    "parameters": {
        "lr": {'values': [1e-2, 1e-3, 1e-4]},
        "step_size": {'values': [3, 5, 7, 10]},
        "gamma": {'values': [0.01, 0.1, 0.25, 0.5]},
    },
}


def train_wrapper():
    with wandb.init() as run:
        train(
            lr=run.config.lr,
            step_size=run.config.step_size,
            gamma=run.config.gamma,
            epochs=50,
            use_wandb=True
        )


sweep_id = wandb.sweep(sweep=sweep_configuration, entity='matteo-ghia-politecnico-di-torino', project="aml challenge 2")
print(f"Sweep ID: {sweep_id}")
wandb.agent(sweep_id, function=train_wrapper)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: matteo-ghia (matteo-ghia-2001) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Create sweep with ID: 3557adrj
Sweep URL: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202/sweeps/3557adrj
Sweep ID: 3557adrj


wandb: Agent Starting Run: dngnte5c with config:
wandb: 	gamma: 0.01
wandb: 	lr: 0.01
wandb: 	step_size: 3
wandb: Currently logged in as: matteo-ghia (matteo-ghia-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.19.9
wandb: Run data is saved locally in /kaggle/working/wandb/run-20250523_134246-dngnte5c
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run dutiful-sweep-1
wandb: ⭐️ View project at https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202
wandb: 🧹 View sweep at https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202/sweeps/3557adrj
wandb: 🚀 View run at https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202/runs/dngnte5c


ROC AUC Score test: 0.5894423637120266
Epoch 1/50, Train Loss: 0.6096, Validation Loss: 0.5072
ROC AUC Score test: 0.5901248439450687
Epoch 2/50, Train Loss: 0.4702, Validation Loss: 0.4288
ROC AUC Score test: 0.6043320848938827
Epoch 3/50, Train Loss: 0.4464, Validation Loss: 0.4222
ROC AUC Score test: 0.61173949230129
Epoch 4/50, Train Loss: 0.4224, Validation Loss: 0.4104
ROC AUC Score test: 0.611926758218893
Epoch 5/50, Train Loss: 0.4178, Validation Loss: 0.4074
ROC AUC Score test: 0.6146899708697462
Epoch 6/50, Train Loss: 0.4159, Validation Loss: 0.4057
ROC AUC Score test: 0.6140241364960466
Epoch 7/50, Train Loss: 0.4152, Validation Loss: 0.4059
ROC AUC Score test: 0.6137910944652517
Epoch 8/50, Train Loss: 0.4150, Validation Loss: 0.4060
ROC AUC Score test: 0.6150603412401165
Epoch 9/50, Train Loss: 0.4148, Validation Loss: 0.4054
ROC AUC Score test: 0.6141843528922181
Epoch 10/50, Train Loss: 0.4148, Validation Loss: 0.4058
ROC AUC Score test: 0.614294631710362
Epoch 11/50, T

wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         epoch ▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
wandb: roc_auc_score ▁▁▅▇▇█████▇██▇███▇████▇██████▇█▇▇██▇███▇
wandb:    train_loss █▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val_loss █▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:         epoch 50
wandb: roc_auc_score 0.61297
wandb:    train_loss 0.41531
wandb:      val_loss 0.40603
wandb: 
wandb: 🚀 View run dutiful-sweep-1 at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202/runs/dngnte5c
wandb: ⭐️ View project at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20250523_134246-dngnte5c/logs
wandb: Agent Starting Run: cavor2k4 with config:
wandb: 	gamma: 0.01
wandb: 	lr: 0.01
wandb: 	step_size: 5
wandb: Trackin

...

wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         epoch ▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
wandb: roc_auc_score ▁▄▄▃▇▇██████████████████████████████████
wandb:    train_loss █▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val_loss █▄▄▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:         epoch 50
wandb: roc_auc_score 0.58414
wandb:    train_loss 0.39786
wandb:      val_loss 0.38457
wandb: 
wandb: 🚀 View run glamorous-sweep-46 at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202/runs/ie3z9hkd
wandb: ⭐️ View project at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20250523_141318-ie3z9hkd/logs
wandb: Agent Starting Run: wwid3zfj with config:
wandb: 	gamma: 0.5
wandb: 	lr: 0.0001
wandb: 	step_size: 7
wandb: Tra

ROC AUC Score test: 0.47868081564710785
Epoch 1/50, Train Loss: 0.8827, Validation Loss: 0.5876
ROC AUC Score test: 0.5001914273824386
Epoch 2/50, Train Loss: 0.5495, Validation Loss: 0.5230
ROC AUC Score test: 0.5016104868913858
Epoch 3/50, Train Loss: 0.5240, Validation Loss: 0.5129
ROC AUC Score test: 0.49494798168955473
Epoch 4/50, Train Loss: 0.5132, Validation Loss: 0.4920
ROC AUC Score test: 0.5141656263004578
Epoch 5/50, Train Loss: 0.4712, Validation Loss: 0.4416
ROC AUC Score test: 0.5439783603828547
Epoch 6/50, Train Loss: 0.4379, Validation Loss: 0.4190
ROC AUC Score test: 0.5651477320016646
Epoch 7/50, Train Loss: 0.4227, Validation Loss: 0.4049
ROC AUC Score test: 0.5745318352059925
Epoch 8/50, Train Loss: 0.4145, Validation Loss: 0.4000
ROC AUC Score test: 0.5752018310445277
Epoch 9/50, Train Loss: 0.4115, Validation Loss: 0.3972
ROC AUC Score test: 0.5776529338327091
Epoch 10/50, Train Loss: 0.4084, Validation Loss: 0.3951
ROC AUC Score test: 0.5779026217228465
Epoch 11

wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         epoch ▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
wandb: roc_auc_score ▁▂▂▂▃▆▇▇▇▇█▇▇▇██████████████████████████
wandb:    train_loss █▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val_loss █▆▆▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:         epoch 50
wandb: roc_auc_score 0.59266
wandb:    train_loss 0.37764
wandb:      val_loss 0.36518
wandb: 
wandb: 🚀 View run quiet-sweep-47 at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202/runs/wwid3zfj
wandb: ⭐️ View project at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20250523_141359-wwid3zfj/logs
wandb: Agent Starting Run: d5jjh7f2 with config:
wandb: 	gamma: 0.5
wandb: 	lr: 0.0001
wandb: 	step_size: 10
wandb: Tracki

ROC AUC Score test: 0.5044111527257594
Epoch 1/50, Train Loss: 0.8763, Validation Loss: 0.5656
ROC AUC Score test: 0.5022846441947566
Epoch 2/50, Train Loss: 0.5419, Validation Loss: 0.5212
ROC AUC Score test: 0.4983853516437786
Epoch 3/50, Train Loss: 0.5245, Validation Loss: 0.5143
ROC AUC Score test: 0.49660424469413234
Epoch 4/50, Train Loss: 0.5193, Validation Loss: 0.5097
ROC AUC Score test: 0.49308364544319605
Epoch 5/50, Train Loss: 0.5145, Validation Loss: 0.5015
ROC AUC Score test: 0.5416354556803995
Epoch 6/50, Train Loss: 0.4770, Validation Loss: 0.4269
ROC AUC Score test: 0.5781315022888056
Epoch 7/50, Train Loss: 0.4215, Validation Loss: 0.3996
ROC AUC Score test: 0.5817686225551394
Epoch 8/50, Train Loss: 0.4103, Validation Loss: 0.3945
ROC AUC Score test: 0.5839658759883479
Epoch 9/50, Train Loss: 0.4070, Validation Loss: 0.3926
ROC AUC Score test: 0.5831772784019975
Epoch 10/50, Train Loss: 0.4047, Validation Loss: 0.3898
ROC AUC Score test: 0.583524760715772
Epoch 11/

wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         epoch ▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
wandb: roc_auc_score ▂▁▁▁▄▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████████████████
wandb:    train_loss █▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:      val_loss █▇▆▆▆▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:         epoch 50
wandb: roc_auc_score 0.60156
wandb:    train_loss 0.3528
wandb:      val_loss 0.34096
wandb: 
wandb: 🚀 View run fresh-sweep-48 at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202/runs/d5jjh7f2
wandb: ⭐️ View project at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%202
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20250523_141440-d5jjh7f2/logs
wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


In [11]:
# model = train(lr=1e-3, weight_decay=1e-8, epochs=50)
# torch.save(model.state_dict(), "model.pt")